URL: https://customer-academy.databricks.com/learn/learning-plans/160/apache-spark-developer-learning-plan/courses/3901/introduction-to-apache-spark/lessons

## Overview


- Spark performance is delivered through in-memory computation which accounts for its speed
- Apache Spark is Open Source and provides a consistent interface across platforms
- Integrates with all major cloud vendors
- DataBricks was founded by the creators of Spark and contributes to the open source development effort still

## Spark Components

![](/Volumes/workspace/pyspark_learning/raw_files/images/spark_components.png)

- Spark Core Engine provides the foundation of the system.  It handles memory mangement, fault tolerance, scheduling, and task distribution
- Spark Core Engine is interacted with via high-level APIs, DataFrame, RDD API, and SQL API
- There are specialized API's which 'sit on top' of the DataFrame API, see image above.

The RDD API is the original, low-level Application Programming Interface in Apache Spark.  **RDD stands for Resilient Distributed Dataset**

## Architecture

- Spark Driver contains the SparkSession/Context, is the 'brain' of the process, issuing instructions
- Cluster Manager manages Spark resources, assigns tasks to the Workers
- Workers are cluster nodes which host the executors
- Executors process the tasks issued by the driver

Driver --> Cluster Mananger --> Worker(s) --> Executor




**Spark Driver**

The primary function of the Spark Driver is:
The Spark Driver analyzes the application and creates a Directed Acyclic Graph (DAG) which is a computer science modeling component.

Further:

The Spark Driver creates the SparkSession and is the entry point for all applications.  When in teh DBX UI and working in a notebook, the SparkSession has been created for you behind the scenes.  If working in VSCode or other, one needs to instanciate their own SparkSession manually. In Databricks the SparkSession is created as 'spark'.  Example Command:


```
spark.createDataFrame(data, schema=schema1)
```

The Driver schedules and distributes task to Executors and then monitors the progess, returning results to the client.


In [0]:
"""
creating my own spark session which I DONT need to do in DBX
Need to import the SparkSession class from the pyspark.sql module
'spark' below is my SparkSession
"""
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

In [0]:
"""
create multiple sessions
"""
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()
spark_1 = SparkSession.builder.getOrCreate()

**Spark DAG**

NOTE:  The acyclic structure ensures no infinite loops are entered

![](/Volumes/workspace/pyspark_learning/raw_files/images/dag.png)



**Relationship between logical plan and DAG**

In Spark, Logical Plans directly map to individual tasks with the DAG without any optimization

**Spark Application Execution**

The application spans 'jobs' which are broken into 'stages' which are then broken into 'tasks'.  Where possible Jobs, stages and tasks are run in parallel, which implies no dependencies between them

## Cluster Types in Databricks

- **All Purpose Clusters**:   these are interactive clusters which support notebooks, jobs, and dashboards.  They are configurable with auto-termination

- **Job Clusters**:   these are ephemeral (short lived) clusters which start when a job runs and terminate automatically when a job finishes.  They are optimized for non-interactive workloads

- **SQL Warehouses**:  Optimized clusters for SQL query performance.   Provides instant startup and auto-scaling

## Distributed Computing in DBX

#### Key Characteristics

- **Independence**: Each node works independently of other nodes and manages own CPU and memory
- **Scalability**: Addition of nodes provides more processing power
- **Fault Tolerance**:  Failures are isolated
- **Resource Partitioning**:  Data and workloads are partitioned across nodes

## Dataframes

**Tungsten** is Sparks columnar in-memory execution engine for consistent representation of dataframes regardless of source data.   Tungsten provides statistics when saving a dataframe to optimized formats (parquet and delta)

## Plans

Spark optimizes the execution plan through several stages (Unresolved Logical Plan to Physical Plan).

**Catalyst Optimizer**:   This engine applies rule-based and cost-based optimizations to convert the DataFrame operations into an optimized execution plan, working in conjunction with the vectorized Photon Engine

**Photon Engine**:  Optimizes query execution processing data in batches rather than row-by-row.  Runs by default in SQL warehouse and serverless compute.  Can be enabled on all-purpose compute and job clusters

Dataframes are immuatable.  Once created they cannot be modified.   Transformations create new dataframs from existing dataframes.

## whats my schema?

```dataframe_name.printShema()```:  this is a method on the schema which will list the columns and datatypes

```print(dataframe_name.schema)```:   this is the pythonic way to display the schema.  The datatypes will be displayed as python StructFields and StructTypes

Inferring a schema has higher overhead as it requires spark jobs to launch.  It has to analyze the data then create the dataframe.

Supplying a schema has lower overhead as no spark jobs are created to process the data, its just a simple transformation

DDL schemas are best for flat data

Python schemas are best for complex, nested data

## Lazy Evaluation

In [0]:
# Lazy evaluation in Spark means that transformations on DataFrames (like filter, select, map) are not executed immediately.
# Instead, Spark builds a logical plan and only executes the computation when an action (like display(), count(), collect()) is called.
# This allows Spark to optimize the execution plan before running it.

# Example:
df = spark.read.format("csv").option("header", "true").load("/databricks-datasets/nyctaxi/taxidata.csv")
filtered_df = df.filter(df["passenger_count"] > 2)  # Transformation (lazy)
display(filtered_df)  # Action (triggers execution)

## Actions vs Transformations

In [0]:
# Spark transformations (e.g., filter, select, map) create a new DataFrame from an existing one and are lazy—they do not trigger computation.
# Spark actions (e.g., display(), count(), collect()) trigger the execution of the transformations and return results or output.

# Example transformation (lazy, does not execute immediately):
transformed_df = df.select("passenger_count").filter(df["passenger_count"] > 2)

# Example action (triggers execution):
display(transformed_df)

### UDFs

UDFs can extend the capabilities of Databricks; however:
- UDFs cannot be optimized by the Catalyst Optimizer and require additional serialization overhead
- Always use a built-in function when possible